# Bilan API — consommation de tokens et coûts

Notebook d'analyse du journal `usage_log.csv` (une ligne par scénario et par
run, écrite par `bench.generate`). **À relancer après chaque génération** :
il est sans état, relisez-le de haut en bas (Run All).

Ce qu'il montre : le dernier run en détail et face à l'historique, l'évolution
run par run (dont la décomposition input/output — l'input trace le poids des
jeux de templates + fiches, l'output la longueur des CRH), le cumul dépensé,
le coût par famille clinique, la part de l'itération (runs partiels), et une
projection à l'échelle.


In [ ]:
# Paramètres et chargement
from pathlib import Path
import polars as pl
import matplotlib
import matplotlib.pyplot as plt

CSV_PATH = Path("usage_log.csv")          # le notebook vit dans work_prompts/
PROJECTIONS = [100, 1_000, 10_000]        # tailles de corpus à projeter

if not CSV_PATH.is_file():
    df = None
    print(f"Pas de journal : {CSV_PATH.resolve()} — il naît au premier run réel.")
else:
    df = pl.read_csv(
        CSV_PATH,
        schema_overrides={"test": pl.Utf8, "scenario": pl.Utf8,
                          "batch_id": pl.Utf8, "out": pl.Utf8},
    ).with_columns(
        pl.col("timestamp_utc").str.to_datetime("%Y-%m-%dT%H:%M:%S%.fZ",
                                                strict=False).alias("ts"),
        (pl.col("input_tokens") + pl.col("output_tokens")).alias("total_tokens"),
    ).sort("ts")
    # Un run = un batch_id ; libellé lisible et ordre chronologique
    runs = (df.group_by("batch_id")
              .agg(pl.col("ts").min().alias("ts"),
                   pl.col("test").first(), pl.col("out").first(),
                   pl.col("partial").first(),
                   pl.len().alias("n_crh"),
                   pl.col("input_tokens").sum(), pl.col("output_tokens").sum(),
                   pl.col("cost_usd").sum())
              .sort("ts")
              .with_columns(
                  (pl.col("test") + "/" + pl.col("out").str.replace(r"\.txt$", "")
                   ).alias("run_label"),
                  (pl.col("cost_usd") / pl.col("n_crh")).alias("cout_par_crh"))
           )
    print(f"{df.height} lignes, {runs.height} runs, "
          f"tests {sorted(df['test'].unique().to_list())}, "
          f"du {df['ts'].min()} au {df['ts'].max()} (UTC)")


In [ ]:
# Hygiène du journal — anomalies à attraper tôt
if df is not None:
    anomalies = df.filter((pl.col("input_tokens") <= 0)
                          | (pl.col("output_tokens") <= 0)
                          | (pl.col("cost_usd") <= 0))
    if anomalies.height:
        print(f"ATTENTION : {anomalies.height} ligne(s) à tokens/coût nuls ou négatifs :")
        print(anomalies.select("test", "out", "scenario", "input_tokens",
                               "output_tokens", "cost_usd"))
    else:
        print("Aucune anomalie (tokens et coûts strictement positifs partout).")
    n_part = df.filter(pl.col("partial") == True).height
    print(f"Lignes de runs partiels : {n_part} / {df.height}")


## Vue d'ensemble — un run par ligne, chronologique

In [ ]:
if df is not None:
    print(runs.select("ts", "run_label", "n_crh", "input_tokens",
                      "output_tokens", "cost_usd", "cout_par_crh", "partial"))


## Dernier run — détail et position face à l'historique

In [ ]:
if df is not None:
    last_id = runs["batch_id"][-1]
    last = runs.filter(pl.col("batch_id") == last_id)
    d_last = df.filter(pl.col("batch_id") == last_id)
    print(f"Dernier run : {last['run_label'][0]}  —  {last['n_crh'][0]} CRH, "
          f"{last['cost_usd'][0]:.4f} $ "
          f"({last['input_tokens'][0]:,} in / {last['output_tokens'][0]:,} out)")

    # Détail par scénario, trié par coût
    print(d_last.select("scenario", "template", "input_tokens", "output_tokens",
                        "cost_usd").sort("cost_usd", descending=True))

    # Barres par scénario, couleur par template
    templates = d_last["template"].unique().sort().to_list()
    cmap = plt.get_cmap("tab10")
    colors = {t: cmap(i % 10) for i, t in enumerate(templates)}
    d_ = d_last.sort("scenario")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.bar(d_["scenario"].to_list(), d_["cost_usd"].to_list(),
           color=[colors[t] for t in d_["template"].to_list()])
    ax.set_title(f"Coût par scénario — {last['run_label'][0]}")
    ax.set_ylabel("USD")
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[t]) for t in templates]
    ax.legend(handles, templates, fontsize=7, ncol=2)
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
# Dernier run vs runs précédents (par CRH, pour comparer à taille inégale)
if df is not None and runs.height >= 2:
    prev = runs.filter(pl.col("batch_id") != last_id)
    prev_test = prev.filter(pl.col("test") == last["test"][0])

    def par_crh(r):
        n = r["n_crh"].sum()
        return (r["cost_usd"].sum() / n, r["input_tokens"].sum() / n,
                r["output_tokens"].sum() / n)

    c_l, i_l, o_l = par_crh(last)
    c_p, i_p, o_p = par_crh(prev)
    print(f"Coût/CRH   : {c_l:.4f} $ (dernier)  vs {c_p:.4f} $ (moy. précédents, "
          f"{100*(c_l-c_p)/c_p:+.1f} %)")
    print(f"Input/CRH  : {i_l:,.0f}  vs {i_p:,.0f}  ({100*(i_l-i_p)/i_p:+.1f} %)"
          "   ← poids des templates + fiches")
    print(f"Output/CRH : {o_l:,.0f}  vs {o_p:,.0f}  ({100*(o_l-o_p)/o_p:+.1f} %)"
          "   ← longueur des CRH")
    if prev_test.height:
        c_t, i_t, o_t = par_crh(prev_test)
        print(f"(même test {last['test'][0]} : coût/CRH précédent {c_t:.4f} $)")
elif df is not None:
    print("Un seul run au journal — la comparaison viendra avec le suivant.")


## Évolution run par run

Deux lectures : le **coût par CRH** (barres), et la **décomposition
input/output par CRH** (aires empilées) — si l'input grimpe au fil des tests,
ce sont les jeux de templates qui grossissent, pas les CRH.

In [ ]:
if df is not None:
    labels = runs["run_label"].to_list()
    x = range(runs.height)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
    a1.bar(x, runs["cout_par_crh"].to_list(),
           color=["#cc7722" if p else "#33658a" for p in runs["partial"].to_list()])
    a1.set_title("Coût par CRH, par run (orange = partiel)")
    a1.set_ylabel("USD / CRH")
    a1.set_xticks(list(x)); a1.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)

    in_crh = (runs["input_tokens"] / runs["n_crh"]).to_list()
    out_crh = (runs["output_tokens"] / runs["n_crh"]).to_list()
    a2.bar(x, in_crh, label="input/CRH (templates+fiches)", color="#7a9e7e")
    a2.bar(x, out_crh, bottom=in_crh, label="output/CRH (texte généré)", color="#2f4858")
    a2.set_title("Tokens par CRH — décomposition")
    a2.set_xticks(list(x)); a2.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()


## Cumul dépensé

In [ ]:
if df is not None:
    cum = runs.with_columns(pl.col("cost_usd").cum_sum().alias("cumul"))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.step(cum["ts"].to_list(), cum["cumul"].to_list(), where="post")
    ax.scatter(cum["ts"].to_list(), cum["cumul"].to_list(), s=14)
    for t, c, l in zip(cum["ts"].to_list(), cum["cumul"].to_list(),
                       cum["run_label"].to_list()):
        ax.annotate(l, (t, c), fontsize=6, xytext=(3, 4),
                    textcoords="offset points")
    ax.set_title(f"Coût cumulé — total {cum['cumul'][-1]:.4f} $")
    ax.set_ylabel("USD")
    plt.tight_layout(); plt.show()


## Par famille clinique

In [ ]:
if df is not None:
    fam = (df.group_by("template")
             .agg(pl.len().alias("n"), pl.col("cost_usd").mean().alias("cout_moyen"),
                  pl.col("input_tokens").mean().alias("in_moyen"),
                  pl.col("output_tokens").mean().alias("out_moyen"))
             .sort("cout_moyen", descending=True))
    print(fam)
    fig, ax = plt.subplots(figsize=(8, 0.35 * fam.height + 1.2))
    ax.barh(fam["template"].to_list()[::-1], fam["cout_moyen"].to_list()[::-1],
            color="#33658a")
    ax.set_title("Coût moyen par CRH selon la famille (tous runs)")
    ax.set_xlabel("USD / CRH")
    plt.tight_layout(); plt.show()


## Coût de l'itération et projection

La part des runs partiels (`only=`, paillasses, reprises) est le prix de
l'expérimentation ; la projection utilise le coût/CRH du **dernier run
complet** — c'est le chiffre à donner pour dimensionner un corpus.

In [ ]:
if df is not None:
    part = runs.filter(pl.col("partial") == True)["cost_usd"].sum()
    total = runs["cost_usd"].sum()
    print(f"Runs partiels : {part:.4f} $ / {total:.4f} $ "
          f"({100*part/total:.1f} % du total)" if total else "—")

    complets = runs.filter(pl.col("partial") == False)
    if complets.height:
        ref = complets["cout_par_crh"][-1]
        print(f"\nProjection au coût du dernier run complet "
              f"({complets['run_label'][-1]}, {ref:.4f} $/CRH) :")
        for n in PROJECTIONS:
            print(f"  {n:>7,} CRH  →  {n*ref:,.2f} $")
    else:
        print("Aucun run complet au journal — projection indisponible.")
